## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# XGBoost
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix
)

# Calibration
from sklearn.calibration import CalibratedClassifierCV

# MLflow
import mlflow
import mlflow.sklearn

# Saving
import joblib
import os

print('Libraries loaded ✓')
print(f'MLflow version: {mlflow.__version__}')

Libraries loaded ✓
MLflow version: 3.10.1


In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Feature names — needed later for Evidently
feature_names = X_train.columns.tolist()

print('Data loaded ✓')
print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feature_names)}')

Data loaded ✓
X_train: (1120, 28)
X_test:  (200, 28)
Features: 28


In [3]:
# Build the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  XGBClassifier(
        n_estimators=107,        # best iteration from Day 3 early stopping
        max_depth=4,
        learning_rate=0.1,
        eval_metric='auc',
        random_state=42,
        verbosity=0
    ))
])

# Fit on training data
pipeline.fit(X_train, y_train)

# Evaluate on test set
y_proba  = pipeline.predict_proba(X_test)[:, 1]
y_pred   = (y_proba >= 0.612).astype(int)   # threshold from Day 3

auc  = roc_auc_score(y_test, y_proba)
f1   = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred)

print('Pipeline trained ✓')
print(f'\nTest Set Performance (threshold=0.612):')
print(f'  ROC-AUC   : {auc:.4f}')
print(f'  F1        : {f1:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')

Pipeline trained ✓

Test Set Performance (threshold=0.612):
  ROC-AUC   : 0.7711
  F1        : 0.5872
  Precision : 0.6531
  Recall    : 0.5333


In [4]:
calibrated_pipeline = CalibratedClassifierCV(
    pipeline,
    method='isotonic',
    cv=5
)

calibrated_pipeline.fit(X_train, y_train)

# Evaluate calibrated pipeline
y_proba_cal = calibrated_pipeline.predict_proba(X_test)[:, 1]
y_pred_cal  = (y_proba_cal >= 0.612).astype(int)

auc_cal  = roc_auc_score(y_test, y_proba_cal)
f1_cal   = f1_score(y_test, y_pred_cal)
prec_cal = precision_score(y_test, y_pred_cal, zero_division=0)
rec_cal  = recall_score(y_test, y_pred_cal)

print('Calibrated pipeline trained ✓')
print(f'\nCalibrated Pipeline — Test Set Performance (threshold=0.612):')
print(f'  ROC-AUC   : {auc_cal:.4f}')
print(f'  F1        : {f1_cal:.4f}')
print(f'  Precision : {prec_cal:.4f}')
print(f'  Recall    : {rec_cal:.4f}')

Calibrated pipeline trained ✓

Calibrated Pipeline — Test Set Performance (threshold=0.612):
  ROC-AUC   : 0.7662
  F1        : 0.5849
  Precision : 0.6739
  Recall    : 0.5167


In [5]:
# Set experiment name — creates it if it doesn't exist
mlflow.set_experiment('german-credit-risk')

# Hyperparameters to log
params = {
    'model'              : 'XGBoost + StandardScaler',
    'n_estimators'       : 107,
    'max_depth'          : 4,
    'learning_rate'      : 0.1,
    'threshold'          : 0.612,
    'calibration_method' : 'isotonic',
    'calibration_cv'     : 5,
    'smote'              : True,
    'train_size'         : len(X_train),
    'test_size'          : len(X_test),
    'n_features'         : X_train.shape[1]
}

# Metrics to log
metrics = {
    'test_roc_auc'  : round(auc_cal,  4),
    'test_f1'       : round(f1_cal,   4),
    'test_precision': round(prec_cal, 4),
    'test_recall'   : round(rec_cal,  4)
}

# Save pipeline locally first — MLflow logs it as artifact
os.makedirs('../models', exist_ok=True)
pipeline_path = '../models/calibrated_pipeline.pkl'
joblib.dump(calibrated_pipeline, pipeline_path)

# Start MLflow run
with mlflow.start_run(run_name='xgb-calibrated-pipeline') as run:

    # Log params
    mlflow.log_params(params)

    # Log metrics
    mlflow.log_metrics(metrics)

    # Log pipeline artifact
    mlflow.log_artifact(pipeline_path, artifact_path='model')

    # Store run ID for later retrieval
    run_id = run.info.run_id

print('MLflow run logged ✓')
print(f'\nRun ID: {run_id}')
print(f'\nParams logged:')
for k, v in params.items():
    print(f'  {k}: {v}')
print(f'\nMetrics logged:')
for k, v in metrics.items():
    print(f'  {k}: {v}')
print(f'\nTo view dashboard: run `mlflow ui` in your terminal from project root')

2026/04/01 00:22:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/01 00:22:09 INFO mlflow.store.db.utils: Updating database tables
2026/04/01 00:22:10 INFO mlflow.tracking.fluent: Experiment with name 'german-credit-risk' does not exist. Creating a new experiment.


MLflow run logged ✓

Run ID: 9cb3f4741e0e4144b59678eb0d7ee376

Params logged:
  model: XGBoost + StandardScaler
  n_estimators: 107
  max_depth: 4
  learning_rate: 0.1
  threshold: 0.612
  calibration_method: isotonic
  calibration_cv: 5
  smote: True
  train_size: 1120
  test_size: 200
  n_features: 28

Metrics logged:
  test_roc_auc: 0.7662
  test_f1: 0.5849
  test_precision: 0.6739
  test_recall: 0.5167

To view dashboard: run `mlflow ui` in your terminal from project root


In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# 1. Retrieve run by ID
run = client.get_run(run_id)

print('Run retrieved ✓')
print(f'\nExperiment ID : {run.info.experiment_id}')
print(f'Run ID        : {run.info.run_id}')
print(f'Status        : {run.info.status}')

# 2. Verify params
print(f'\nLogged params:')
for k, v in run.data.params.items():
    print(f'  {k}: {v}')

# 3. Verify metrics
print(f'\nLogged metrics:')
for k, v in run.data.metrics.items():
    print(f'  {k}: {v}')

# 4. Load artifact and verify predictions match
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path='model/calibrated_pipeline.pkl'
)

loaded_pipeline = joblib.load(artifact_path)
y_proba_loaded  = loaded_pipeline.predict_proba(X_test)[:, 1]

# Check predictions are identical
predictions_match = np.allclose(y_proba_cal, y_proba_loaded)
print(f'\nArtifact loaded ✓')
print(f'Predictions match original: {predictions_match} ✅' if predictions_match
      else 'Predictions do NOT match ❌ — investigate')

Run retrieved ✓

Experiment ID : 1
Run ID        : 9cb3f4741e0e4144b59678eb0d7ee376
Status        : FINISHED

Logged params:
  model: XGBoost + StandardScaler
  n_estimators: 107
  max_depth: 4
  learning_rate: 0.1
  threshold: 0.612
  calibration_method: isotonic
  calibration_cv: 5
  smote: True
  train_size: 1120
  test_size: 200
  n_features: 28

Logged metrics:
  test_roc_auc: 0.7662
  test_f1: 0.5849
  test_precision: 0.6739
  test_recall: 0.5167

Artifact loaded ✓
Predictions match original: True ✅


In [ ]:
# from evidently.future.report import Report
# from evidently.future.presets import DataDriftPreset

# # Reference = training portion of original data (before SMOTE)
# # Current = test set
# df_engineered = pd.read_csv('../data/processed/german_credit_engineered.csv')

# reference = df_engineered.drop(columns=['target']).iloc[:800]
# current   = X_test.copy()

# # Align columns
# common_cols = [c for c in reference.columns if c in current.columns]
# reference   = reference[common_cols]
# current     = current[common_cols]

# print(f'Reference set : {reference.shape}')
# print(f'Current set   : {current.shape}')
# print(f'Features compared: {len(common_cols)}')

# # Build and run the drift report
# report = Report([DataDriftPreset()])

# report.run(
#     reference_data=reference,
#     current_data=current
# )
# print([m for m in dir(report) if not m.startswith('_')])
# # Save HTML report
# os.makedirs('../reports', exist_ok=True)
# report_path = '../reports/data_drift_report.html'
# report.save(report_path)

# print(f'\nDrift report saved to: {report_path} ✓')
# print('Open this file in your browser to see the full interactive report')

Reference set : (800, 28)
Current set   : (200, 28)
Features compared: 28
['include_tests', 'items', 'metadata', 'metrics', 'run', 'set_batch_size', 'set_dataset_id', 'set_model_id', 'set_reference_id', 'tags']


AttributeError: 'Report' object has no attribute 'save'

In [12]:
from scipy.stats import ks_2samp, chi2_contingency

df_engineered = pd.read_csv('../data/processed/german_credit_engineered.csv')

reference = df_engineered.drop(columns=['target']).iloc[:800]
current   = X_test.copy()

common_cols = [c for c in reference.columns if c in current.columns]
reference   = reference[common_cols]
current     = current[common_cols]

# Classify features
binary_cols     = [c for c in common_cols if reference[c].nunique() == 2]
continuous_cols = [c for c in common_cols if reference[c].nunique() > 2]

drift_results = []

# KS test for continuous features
for col in continuous_cols:
    stat, p = ks_2samp(reference[col], current[col])
    drift_results.append({
        'feature' : col,
        'test'    : 'KS',
        'stat'    : round(stat, 4),
        'p_value' : round(p, 4),
        'drifted' : p < 0.05
    })

# Chi-square for binary features
for col in binary_cols:
    ref_counts = reference[col].value_counts().sort_index()
    cur_counts = current[col].value_counts().reindex(ref_counts.index, fill_value=0)
    stat, p, _, _ = chi2_contingency(
        pd.DataFrame([ref_counts.values, cur_counts.values])
    )
    drift_results.append({
        'feature' : col,
        'test'    : 'Chi2',
        'stat'    : round(stat, 4),
        'p_value' : round(p, 4),
        'drifted' : p < 0.05
    })

drift_df = pd.DataFrame(drift_results).sort_values('p_value')

drifted     = drift_df[drift_df['drifted'] == True]
not_drifted = drift_df[drift_df['drifted'] == False]

print(f'Total features tested : {len(drift_df)}')
print(f'Drifted (p < 0.05)    : {len(drifted)}')
print(f'Stable  (p >= 0.05)   : {len(not_drifted)}')

print(f'\n--- Drifted Features ---')
print(drifted[['feature', 'test', 'stat', 'p_value']].to_string(index=False))

print(f'\n--- Stable Features (top 10) ---')
print(not_drifted[['feature', 'test', 'stat', 'p_value']].head(10).to_string(index=False))

# Save results
os.makedirs('../reports', exist_ok=True)
drift_df.to_csv('../reports/drift_report.csv', index=False)
print(f'\nDrift report saved to: ../reports/drift_report.csv ✓')

Total features tested : 28
Drifted (p < 0.05)    : 0
Stable  (p >= 0.05)   : 28

--- Drifted Features ---
Empty DataFrame
Columns: [feature, test, stat, p_value]
Index: []

--- Stable Features (top 10) ---
                                      feature test   stat  p_value
                 personal_status_male mar/wid Chi2 1.2629   0.2611
                      other_parties_guarantor Chi2 1.1174   0.2905
                  personal_status_male single Chi2 1.0692   0.3011
                 credit_history_existing paid Chi2 0.8753   0.3495
                                          age   KS 0.0700   0.4024
                           other_parties_none Chi2 0.5350   0.4645
credit_history_critical/other existing credit Chi2 0.5124   0.4741
                               savings_status   KS 0.0625   0.5473
            property_magnitude_life insurance Chi2 0.2240   0.6360
                       installment_commitment   KS 0.0562   0.6795

Drift report saved to: ../reports/drift_report.csv ✓


## Day 4 — Summary

### What was built today

#### sklearn Pipeline
- StandardScaler + XGBClassifier wrapped into a single Pipeline object
- CalibratedClassifierCV wraps the full pipeline — scaling + prediction + calibration in one call
- Saved to: `../models/calibrated_pipeline.pkl`
- Day 6 Streamlit only needs to call `pipeline.predict_proba(input)` — no manual preprocessing

#### MLflow Experiment Tracking
- Experiment: `german-credit-risk`
- Run ID: `9cb3f4741e0e4144b59678eb0d7ee376`
- Logged: 11 params, 4 metrics, 1 artifact
- Verified: artifact loads correctly, predictions match exactly
- To view dashboard: `mlflow ui` from project root → localhost:5000

#### Data Drift Analysis
- 28 features tested — 0 drifted
- Continuous features: Kolmogorov-Smirnov test
- Binary features: Chi-square test
- Result confirms stratified split preserved distribution correctly
- Report saved to: `../reports/drift_report.csv`

### Final metrics carried forward
| Metric    | Value  |
|-----------|--------|
| ROC-AUC   | 0.7662 |
| F1        | 0.5849 |
| Precision | 0.6739 |
| Recall    | 0.5167 |
| Threshold | 0.612  |

### What Day 5 builds on
- SHAP values computed on the XGBoost model inside the pipeline
- Feature importance ties back to Day 1 EDA findings
- Error profiles from Day 3 explained by SHAP waterfall plots
- financial_stress_score expected to be the top SHAP feature